In [45]:
import sqlite3
import pandas as pd
import numpy as np
conn = sqlite3.connect('../laptops.db')

sql_query = """
    select L.Company, S.ScreenResolution, S.Inches, S.Cpu, S.Ram, S.Memory, S.Gpu, S.Weight, S.OpSys, S.Price
    from Laptops L
    join Specs S on L.laptop_id = S.laptop_id
"""
df = pd.read_sql_query(sql_query, conn)


In [46]:
df_org = pd.read_sql_query(sql_query, conn)

In [47]:
# ram
df['Ram'] = df['Ram'].str.replace('GB', '')
df['Ram'] = df['Ram'].astype(int)

In [48]:
# memory 
drive_type = ['SSD', 'HDD', 'Flash Storage', 'Hybrid']
def memory_extract(column, drive):
    extracted = column.str.extract(rf'(\d+)(GB|TB)\s+{drive}')
    size = pd.to_numeric(extracted[0])
    multiplier = extracted[1].map({'GB':1, 'TB':1024})
    df[f'{drive}_Size'] = (size * multiplier).fillna(0).astype(int)

for drive in drive_type:
    memory_extract(df['Memory'], drive)

In [49]:
#processing (weight)
df['Weight'] = df['Weight'].str.replace('kg', '')
df['Weight'] = df['Weight'].astype(float)

In [61]:
df.head(10)

,Ram,Weight,OpSys,Price,SSD_Size,HDD_Size,Flash Storage_Size,Hybrid_Size,TouchScreen,IPS,...,Company_Apple,Company_Asus,Company_Dell,Company_HP,Company_Lenovo,Company_MSI,Company_Other,Company_Toshiba,Cpu brand,Gpu_brand
0,8,1.37,MacOS,71378.6832,128,0,0,0,0,1,...,1,0,0,0,0,0,0,0,Intel Core i5,Intel
1,8,1.34,MacOS,47895.5232,0,0,128,0,0,0,...,1,0,0,0,0,0,0,0,Intel Core i5,Intel
2,8,1.86,Others/No OS/Linux,30636.0000,256,0,0,0,0,0,...,0,0,0,1,0,0,0,0,Intel Core i5,Intel
3,16,1.83,MacOS,135195.3360,512,0,0,0,0,1,...,1,0,0,0,0,0,0,0,Intel Core i7,AMD
4,8,1.37,MacOS,96095.8080,256,0,0,0,0,1,...,1,0,0,0,0,0,0,0,Intel Core i5,Intel
5,4,2.10,Windows,21312.0000,0,500,0,0,0,0,...,0,0,0,0,0,0,0,0,AMD Processor,AMD
6,16,2.04,MacOS,114017.6016,0,0,256,0,0,1,...,1,0,0,0,0,0,0,0,Intel Core i7,Intel
7,8,1.34,MacOS,61735.5360,0,0,256,0,0,0,...,1,0,0,0,0,0,0,0,Intel Core i5,Intel
8,16,1.30,Windows,79653.6000,512,0,0,0,0,0,...,0,1,0,0,0,0,0,0,Intel Core i7,Nvidia
9,8,1.60,Windows,41025.6000,256,0,0,0,0,1,...,0,0,0,0,0,0,0,0,Intel Core i5,Intel


In [50]:
# screen resolution 
df['TouchScreen'] = df['ScreenResolution'].str.contains('Touchscreen', case=False).astype(int)
df['IPS'] = df['ScreenResolution'].str.contains('IPS', case=False).astype(int)

In [51]:
xy_resolution = df['ScreenResolution'].str.extract(r'(\d+)x(\d+)')
x_res = xy_resolution[0].astype(int)
y_res = xy_resolution[1].astype(int)
df['PPI'] = np.sqrt((x_res)**2 + (y_res)**2) / df['Inches'].astype(float)

In [52]:
# company
count = df['Company'].value_counts()
other = count[count <= 10].index
df['Company'] = df['Company'].replace(other, 'Other')

In [53]:
df = pd.get_dummies(df, columns=['Company'], drop_first=True, dtype=int)

In [54]:
df = df.drop(columns=['ScreenResolution', 'Inches', 'Memory'])

In [55]:
def fetch_cpu(text):
    words = text.split()
    header = " ".join(words[:3])
    if header == 'Intel Core i7' or header == 'Intel Core i5' or header == 'Intel Core i3':
        return header
    else:
        if words[0] == 'Intel':
            return 'Other Intel Processor'
        else:
            return 'AMD Processor'
        
df['Cpu brand'] = df['Cpu'].apply(fetch_cpu)

In [56]:
df['Gpu_brand'] = df['Gpu'].apply(lambda x: x.split()[0])
df = df[df['Gpu_brand'] != 'ARM']

In [57]:
df = df.drop(columns=['Cpu','Gpu'])

In [60]:
def catOS(text):
    if "Windows" in text:
        return "Windows"
    elif "Mac" in text or "mac" in text:
        return "MacOS"
    else:
        return 'Others/No OS/Linux'

df['OpSys'] = df['OpSys'].apply(catOS)